### Load environment variables

In this step, we use the dotenv package to load environment variables from a .env file. This approach helps manage sensitive configuration details like API keys and service credentials without hardcoding them in the code.

Key Points:

dotenv: Automatically loads environment variables defined in a .env file into process.env.

Access Environment Variables: The process.env object is used to access these variables in the application.

In [ ]:
import dotenv from 'dotenv';
dotenv.config();
 
console.log(process.env.AICORE_SERVICE_KEY); 

### Create a New Orchestration Configuration
In this step, we define a function to create an orchestration configuration using the ConfigurationApi from the SAP AI SDK. This configuration integrates various parameters needed for orchestration, such as the executable ID and scenario ID.

Key Points:

ConfigurationApi: Provides methods for interacting with the SAP AI SDK's configuration services.

parameterBindings: Specifies the parameters used for orchestration.

In [2]:
import { ConfigurationApi } from '@sap-ai-sdk/ai-api';

// Function to create orchestration configuration
async function createOrchestrationConfiguration() {
  const requestBody = {
      name: 'orchestration-config', // Choose a meaningful name
      executableId: 'orchestration', // Orchestration executable ID
      scenarioId: 'orchestration', // Orchestration scenario ID
      parameterBindings: [
          {
              "key": "modelFilterList", // Define the parameters you need for orchestration
              "value": "null"  // Example orchestration version
          },
          {
              "key": "modelFilterListType",
              "value": "allow"
          }
      ],
      inputArtifactBindings: []  // Orchestrations may not require input bindings directly, but this can be modified
  };
  try {
      const responseData = await ConfigurationApi
          .configurationCreate(requestBody, {'AI-Resource-Group': 'default'}) // Use the correct resource group
          .execute();
      
      console.log('Orchestration configuration created successfully:', responseData);
      return responseData; // Return the configuration response
  } catch (errorData) {
      const apiError = errorData.response.data.error; // Handle API errors
      console.error('Status code:', errorData.response.status);
      throw new Error(`Configuration creation failed: ${apiError.message}`);
  }
}

In [ ]:
// usage
const orchestrationConfig = await createOrchestrationConfiguration();
orchestrationConfig;

### Deployment of orchestration
This step involves creating a deployment using the specified configuration and resource group. The deployment is handled via the DeploymentApi, which streamlines the process of activating the orchestration setup.

Key Points:

DeploymentApi: Used for initiating the deployment based on the given configuration.

createDeployment Function: This function handles the API call to create the deployment.

In [4]:
import { DeploymentApi } from '@sap-ai-sdk/ai-api';
import type { AiDeploymentCreationResponse } from '@sap-ai-sdk/ai-api';

/**
 * Create a deployment using the configuration specified by configurationId.
 * @param configurationId - ID of the configuration to be used.
 * @param resourceGroup - AI-Resource-Group where the resources are available.
 * @returns Deployment creation response with 'targetStatus': 'RUNNING'.
 */
export async function createDeployment(
  configurationId: string,
  resourceGroup: string
): Promise<AiDeploymentCreationResponse> {
  return DeploymentApi.deploymentCreate(
    { configurationId },
    { 'AI-Resource-Group': resourceGroup }
  ).execute();
}
/**
 * Deploy the orchestration using the given configuration ID.
 * @param resourceGroup - AI-Resource-Group where the resources are available.
 * @returns A message indicating the result of the deployment operation.
 */
export async function deployOrchestration(
  resourceGroup: string
): Promise<string> {
  // Fetch the configuration ID (can be retrieved or passed dynamically)
   const configurationId = orchestrationConfig.id;

  try {
    // Step: Create deployment using the configuration ID
    const response = await createDeployment(configurationId, resourceGroup);
    // console.log(`Orchestration deployment created with ID: ${response.id}`);
    return `Orchestration deployment created with ID: ${response.id}`;
  } catch (error) {
    console.error('Error creating orchestration deployment:', error);
    return 'Failed to create orchestration deployment.';
  }
}

In [ ]:
// usage to deploy orchestration
(async () => {
    const resourceGroup = 'default'; // Replace with your actual resource group name
    try {
      const result = await deployOrchestration(resourceGroup);
      console.log(result); // Outputs deployment creation response
    } catch (error) {
      console.error('Error executing orchestration deployment:', error);
    }
  })();

### Basic Orchestration Pipeline

In [6]:
const filePath = './cv.txt';
let txtContent; 
try {
     txtContent = await Deno.readTextFile(filePath);
    console.log(txtContent);
} catch (error) {
    console.error('Error reading the file:', error);
}
console.log(txtContent);

Gaurav jain
1234 Data St, San Francisco, CA 94101
(123) 456-7890
johndoe@email.com
LinkedIn Profile
GitHub Profile

Objective
Detail-oriented Data Scientist with 3+ years of experience in data analysis, statistical modeling, and machine learning. Seeking to leverage expertise in predictive modeling and data visualization to help drive data-informed decision-making at [Company Name].

Education
Master of Science in Data Science
University of California, Berkeley
Graduated: May 2021

Bachelor of Science in Computer Science
University of California, Los Angeles
Graduated: May 2019

Technical Skills

Programming Languages: Python, R, SQL, Java
Data Analysis & Visualization: Pandas, NumPy, Matplotlib, Seaborn, Tableau
Machine Learning: Scikit-learn, TensorFlow, Keras, XGBoost
Big Data Technologies: Hadoop, Spark
Databases: MySQL, PostgreSQL
Version Control: Git

Professional Experience

Data Scientist
DataCorp Inc., San Francisco, CA
June 2021 – Present

Developed predictive models to optim

### Templating

Explanation of Templating Code

This code defines a template for an AI assistant using orchestration configuration. The `Template` object is set up with system and user messages to guide the assistant’s response behavior. 

Key Components:
- **SystemMessage**: Sets a predefined instruction for the AI assistant. This message typically includes the assistant's role and any specific guidelines it should follow.
- **UserMessage**: Represents the user's input and how it is structured in the conversation.
  
In this revised prompt, only queries are passed to the assistant without any additional context. The AI is expected to respond based solely on the provided input.


In [7]:

// Define the template for resume screening
const templateConfig = {
  templating: {
    template: [
      {
        role: 'system',
        content: 'You are an AI assistant designed to screen resumes for HR purposes. Please assess the candidate qualifications based on the provided resume.',
      },
      {
        role: 'user',
        content: 'Candidate Resume:\n{{?candidate_resume}}',
      },
    ],
  },
};

console.log('Resume screening template configuration defined successfully.');


Resume screening template configuration defined successfully.


###  Define the LLM 

The LLM class is used to configure and initialize a model for generating text based on specific parameters. In this example, we'll use the model to perform the content creation task.

ℹ️Note that virtual deployment of the model is managed automatically by the Orchestration Service, so no additional deployment setup is required on your part.

In [8]:
// List of models to iterate through
const modelName = 'gpt-4o';

In [9]:
const modelConfig = {
  llm: {
    model_name: modelName,
    model_params: {
      max_tokens: 1000,
      temperature: 0.6,
    },
  },
  ...templateConfig,
};
const deploymentConfig = {
  resourceGroup: 'default',
};


### Generate Responses 

This step outlines the process of generating responses for a set of queries using a defined model. The generateResponses function executes queries to gather AI-generated responses.

Key Points:

Query Execution: Uses OrchestrationClient to generate responses for each query.

In [ ]:

import {OrchestrationClient} from '@sap-ai-sdk/orchestration';

const orchestrationClient = new OrchestrationClient({
  ...deploymentConfig,
  ...modelConfig,
});
export async function generateResponse(txtContent: string) {
  try {
    const response = await orchestrationClient.chatCompletion({
      inputParams: { candidate_resume: txtContent },
    });

    const content = response.getContent();

    console.log(`\n=== Response from ${modelName} ===\n`);
    console.log(content);
    console.log('='.repeat(80));
  } catch (error: any) {
    console.error(
      `Error while calling ${modelName}:`,
      error.response?.data || error.message,
    );
  }
}

generateResponse(txtContent);

Promise { <pending> }


=== Response from gpt-4o ===

The candidate, Gaurav Jain, presents a strong resume with relevant qualifications for a data scientist role. Here is an assessment based on the information provided:

**Strengths:**
1. **Education:** Gaurav holds a Master of Science in Data Science from a reputable institution (UC Berkeley) and a Bachelor of Science in Computer Science, providing a solid education foundation in relevant fields.
2. **Technical Skills:** The candidate lists a comprehensive set of technical skills, including programming languages (Python, R, SQL, Java), data analysis and visualization tools (Pandas, NumPy, Matplotlib, Seaborn, Tableau), machine learning frameworks (Scikit-learn, TensorFlow, Keras, XGBoost), big data technologies (Hadoop, Spark), and databases (MySQL, PostgreSQL). This demonstrates versatility and proficiency in key areas required for data science.
3. **Professional Experience:** Gaurav has over 3 years of experience as a data scientist, with achievements suc